# Preprocessing for the modelling notebook

Turns `training_v2.csv` into the model-ready splits in `data/preprocessed/12-01/`, keeping only the 25 features Lasso selected (plus `hospital_id` and `icu_id`).

Order matters here, and it's chosen to keep the test set clean:

1. **Indicate + impute.** Add missingness flags, then IterativeImputer (numeric) / mode (categorical).
2. **Split.** 64 / 16 / 20 train / val / test, stratified on `hospital_death`.
3. **Scale.** Fit on train only.
4. **SMOTENC.** Oversample deaths to 50/50, **training data only**. Val and test keep the real 8.6% prevalence.
5. **Square + one-hot.** Add the Goldilocks squared terms.

`*_dt.csv` files are the pre-scaling, pre-SMOTE versions used by the decision tree.

> Caveat: step 1 is fit on the full dataset before splitting, so the imputer has seen test rows. It's unsupervised and the effect is likely small, but a stricter pipeline would fit it on train only.


In [2]:
import os
import re
from collections import defaultdict
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline, FeatureUnion
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin


In [3]:
class SquaredFeaturesOnly(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        self.feature_names_in_ = getattr(
            X, "columns", [f"x{i}" for i in range(X.shape[1])]
        )
        return self

    def transform(self, X):
        # Convert to numpy if pandas
        X_array = X.values if hasattr(X, "values") else X

        # Add only squared terms, no interactions
        X_squared = X_array**2

        # Combine original and squared
        return np.hstack([X_array, X_squared])

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_

        # Original feature names + squared feature names
        original_names = list(input_features)
        squared_names = [f"{name}^2" for name in input_features]

        return np.array(original_names + squared_names)


In [4]:
SEED = 42


#### Load Data


In [5]:
df = pd.read_csv("../data/training_v2.csv")
df.head()


,encounter_id,patient_id,hospital_id,hospital_death,age,bmi,elective_surgery,ethnicity,gender,height,...,aids,cirrhosis,diabetes_mellitus,hepatic_failure,immunosuppression,leukemia,lymphoma,solid_tumor_with_metastasis,apache_3j_bodysystem,apache_2_bodysystem
0,66154,25312,118,0,68.0,22.73,0,Caucasian,M,180.3,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Sepsis,Cardiovascular
1,114252,59342,81,0,77.0,27.42,0,Caucasian,F,160.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Respiratory,Respiratory
2,119783,50777,118,0,25.0,31.95,0,Caucasian,F,172.7,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Metabolic,Metabolic
3,79267,46918,118,0,81.0,22.64,1,Caucasian,F,165.1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Cardiovascular,Cardiovascular
4,92056,34377,33,0,19.0,NaN,0,Caucasian,M,188.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Trauma,Trauma


In [6]:
# Load metadata
df_meta = pd.read_csv("../data/WiDS Datathon 2020 Dictionary.csv")
df_meta = df_meta[~df_meta["Variable Name"].isin(["icu_admit_type", "pred"])]


In [7]:
# Define column types based on metadata
pre_col_types = {}
pre_types_col = defaultdict(set)

for _, row in df_meta.iterrows():
    pre_types_col[row["Data Type"]].add(row["Variable Name"])
    pre_col_types[row["Variable Name"]] = row["Data Type"]

pre_col_types["bmi"] = "numeric"
pre_types_col["numeric"].add("bmi")
pre_types_col["string"].remove("bmi")

pre_num_cols = set()
pre_cat_cols = set()
pre_bin_cols = set()
for col in df.columns:
    if pre_col_types[col] in ("integer", "numeric"):
        pre_num_cols.add(col)
    elif pre_col_types[col] == "binary":
        pre_bin_cols.add(col)
    else:
        pre_cat_cols.add(col)
pre_all_cols = pre_num_cols | pre_bin_cols | pre_cat_cols


#### Load Features


In [8]:
lasso_coeff = pd.read_csv("../data/lasso_coeff.csv")
lasso_coeff.head()


,Lambda,poly_goldilocks__d1_heartrate_max,poly_goldilocks__d1_heartrate_min,poly_goldilocks__d1_sysbp_min,poly_goldilocks__d1_sysbp_max,poly_goldilocks__d1_mbp_min,poly_goldilocks__d1_mbp_max,poly_goldilocks__d1_temp_min,poly_goldilocks__d1_temp_max,poly_goldilocks__d1_resprate_min,...,remainder__aids_missing,remainder__cirrhosis_missing,remainder__diabetes_mellitus_missing,remainder__hepatic_failure_missing,remainder__immunosuppression_missing,remainder__leukemia_missing,remainder__lymphoma_missing,remainder__solid_tumor_with_metastasis_missing,remainder__apache_3j_bodysystem_missing,remainder__apache_2_bodysystem_missing
0,0.001000,0.227917,0.152262,0.0,-0.085112,-0.141534,-0.066575,0.0,-0.011664,0.108256,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.001374,0.219238,0.149404,0.0,-0.077578,-0.144545,-0.062695,0.0,-0.009811,0.106944,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.001887,0.211432,0.141043,0.0,-0.066228,-0.147464,-0.059073,0.0,-0.007358,0.105874,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.002593,0.193675,0.131073,0.0,-0.055041,-0.152455,-0.052783,0.0,0.000000,0.105215,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.003562,0.172800,0.115552,0.0,-0.040307,-0.155607,-0.044394,0.0,0.000000,0.103261,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Will use 26 features from Lambda index 10
(lasso_coeff.drop(columns=["Lambda"]) != 0).sum(axis=1)


0     129
1     116
2     106
3      88
4      68
5      62
6      55
7      46
8      42
9      32
10     25
11     20
12     16
13     16
14     12
15      9
16      4
17      3
18      2
19      2
20      2
21      0
22      0
23      0
24      0
25      0
26      0
27      0
28      0
29      0
dtype: int64

In [10]:
lasso_coeff.iloc[10]["Lambda"]


np.float64(0.0239502661998748)

In [11]:
lambda_index = 10

lasso_features: list[str] = (
    lasso_coeff.loc[lambda_index]
    .drop(["Lambda"])[lasso_coeff.loc[lambda_index].drop(["Lambda"]) != 0]
    .index.tolist()
)

print(lasso_features)

lasso_features = [re.sub(r"^.*__", "", feature) for feature in lasso_features]

lasso_features


['poly_goldilocks__d1_heartrate_max', 'poly_goldilocks__d1_mbp_min', 'poly_goldilocks__d1_resprate_min', 'poly_goldilocks__d1_spo2_min', 'poly_goldilocks__age', 'poly_goldilocks__d1_heartrate_min^2', 'poly_goldilocks__d1_sysbp_min^2', 'poly_goldilocks__d1_temp_min^2', 'poly_goldilocks__d1_temp_max^2', 'poly_goldilocks__d1_resprate_min^2', 'poly_goldilocks__d1_potassium_min^2', 'poly_goldilocks__d1_glucose_max^2', 'poly_goldilocks__d1_glucose_min^2', 'poly_goldilocks__d1_sodium_max^2', 'poly_goldilocks__d1_wbc_min^2', 'poly_goldilocks__d1_calcium_min^2', 'cat__ventilated_apache_1.0', 'num__resprate_apache', 'num__h1_resprate_max', 'num__gcs_eyes_apache', 'num__h1_resprate_min', 'num__d1_sysbp_noninvasive_min', 'num__gcs_verbal_apache', 'num__d1_bun_min', 'num__gcs_motor_apache']


['d1_heartrate_max',
 'd1_mbp_min',
 'd1_resprate_min',
 'd1_spo2_min',
 'age',
 'd1_heartrate_min^2',
 'd1_sysbp_min^2',
 'd1_temp_min^2',
 'd1_temp_max^2',
 'd1_resprate_min^2',
 'd1_potassium_min^2',
 'd1_glucose_max^2',
 'd1_glucose_min^2',
 'd1_sodium_max^2',
 'd1_wbc_min^2',
 'd1_calcium_min^2',
 'ventilated_apache_1.0',
 'resprate_apache',
 'h1_resprate_max',
 'gcs_eyes_apache',
 'h1_resprate_min',
 'd1_sysbp_noninvasive_min',
 'gcs_verbal_apache',
 'd1_bun_min',
 'gcs_motor_apache']

In [12]:
no_ohe_cat_features = ["hospital_id", "icu_id"]


In [13]:
pre_features = set()
num_features = []
cols_to_square = []
cols_to_miss_indicate = []
cat_features_dict = defaultdict(set)
bin_features = set()

for feature in lasso_features:
    for col in pre_all_cols:
        if re.match(rf"^{col}", feature):
            pre_features.add(col)
            if col == feature:
                # Numeric feature
                num_features.append(feature)
            elif feature[len(col)] == "^":
                # Polynomial (^2) numeric feature
                num_features.append(feature)
                cols_to_square.append(col)
            elif feature == f"{col}_missing":
                # Missing indicator feature
                cols_to_miss_indicate.append(col)
                bin_features.add(f"missingindicator_{col}")
            elif re.match(rf"^{col}_[01](.0)?$", feature):
                # Binary feature
                bin_features.add(col)
            else:
                # Categorical feature
                category = pd.Series(feature[len(col) + 1 :], dtype=df[col].dtype)[0]
                cat_features_dict[col].add(category)
            break

pre_features = list(pre_features)
pre_bin_cols = [col for col in pre_features if col in pre_bin_cols]
pre_cat_cols = [col for col in pre_features if col in pre_cat_cols]
pre_num_cols = [col for col in pre_features if col in pre_num_cols]

cat_features = list(cat_features_dict.keys())
bin_features = list(bin_features)

cat_features_categories = [list(cat_features_dict[col]) for col in cat_features]

features = num_features + bin_features + cat_features


#### 1. Indicate, Impute


In [14]:
preprocessing_1 = ImbPipeline(
    steps=[
        (
            "indicator",
            ColumnTransformer(
                transformers=[
                    (
                        "original_and_missing",
                        FeatureUnion(
                            transformer_list=[
                                ("original", "passthrough"),
                                ("missing_indicator", MissingIndicator()),
                            ],
                            verbose_feature_names_out=False,
                        ),
                        cols_to_miss_indicate,
                    )
                ],
                remainder="passthrough",
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
        (
            "imputers",
            ColumnTransformer(
                transformers=[
                    (
                        "num_imputer",
                        IterativeImputer(max_iter=100, random_state=SEED),
                        pre_num_cols,
                    ),
                    (
                        "cat_imputer",
                        SimpleImputer(strategy="most_frequent"),
                        pre_cat_cols + pre_bin_cols,
                    ),
                ],
                remainder="passthrough",
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
    ]
).set_output(transform="pandas")

df_preprocessed_1 = pd.concat(
    [df[no_ohe_cat_features], preprocessing_1.fit_transform(df[pre_features])], axis=1
)


#### 2. Split


In [15]:
X_train_val_2, X_test_2, y_train_val_2, y_test_2 = train_test_split(
    df_preprocessed_1,
    df["hospital_death"],
    test_size=0.2,
    random_state=SEED,
    stratify=df["hospital_death"],
)

X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(
    X_train_val_2,
    y_train_val_2,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train_val_2,
)


#### 3. Scale


In [16]:
preprocessing_3 = ColumnTransformer(
    transformers=[
        (
            "num_scaler",
            StandardScaler(),
            pre_num_cols,
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X_train_val_3 = preprocessing_3.fit_transform(X_train_val_2)
X_test_3 = preprocessing_3.transform(X_test_2)

X_train_3 = preprocessing_3.fit_transform(X_train_2)
X_val_3 = preprocessing_3.transform(X_val_2)


y_train_val_3 = y_train_val_2
y_test_3 = y_test_2

y_train_3 = y_train_2
y_val_3 = y_val_2


#### 4. Sample/Balance


In [17]:
X_train_val_4, y_train_val_4 = SMOTENC(
    categorical_features=cat_features + bin_features + no_ohe_cat_features,
    random_state=SEED,
).fit_resample(X_train_val_3, y_train_val_3)

X_train_4, y_train_4 = SMOTENC(
    categorical_features=cat_features + bin_features + no_ohe_cat_features,
    random_state=SEED,
).fit_resample(X_train_3, y_train_3)


X_test_4, y_test_4 = X_test_3, y_test_3

X_val_4, y_val_4 = X_val_3, y_val_3


#### 5. Polynomial Terms, One-Hot Encoding


In [18]:
preprocessing_5 = ImbPipeline(
    steps=[
        (
            "poly_features",
            ColumnTransformer(
                transformers=[
                    (
                        "poly_transformer",
                        SquaredFeaturesOnly(),
                        cols_to_square,
                    )
                ],
                remainder="passthrough",
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
        (
            "encoder",
            ColumnTransformer(
                transformers=[
                    (
                        "onehot_encoder",
                        OneHotEncoder(
                            categories=cat_features_categories,
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                        cat_features,
                    )
                ],
                remainder="passthrough",
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
    ]
).set_output(transform="pandas")

X_train_val_5 = preprocessing_5.fit_transform(X_train_val_4)
X_test_5 = preprocessing_5.transform(X_test_4)
X_train_5 = preprocessing_5.transform(X_train_4)
X_val_5 = preprocessing_5.transform(X_val_4)

y_train_val_5 = y_train_val_4
y_test_5 = y_test_4
y_train_5 = y_train_4
y_val_5 = y_val_4


In [19]:
X_train_val = X_train_val_5
y_train_val = y_train_val_5

X_test = X_test_5
y_test = y_test_5


X_train = X_train_5
y_train = y_train_5

X_val = X_val_5
y_val = y_val_5


In [20]:
assert (
    X_train_val.columns.tolist()
    == X_test.columns.tolist()
    == X_train.columns.tolist()
    == X_val.columns.tolist()
)


In [21]:
X_train.columns.tolist()


['d1_heartrate_min',
 'd1_sysbp_min',
 'd1_temp_min',
 'd1_temp_max',
 'd1_resprate_min',
 'd1_potassium_min',
 'd1_glucose_max',
 'd1_glucose_min',
 'd1_sodium_max',
 'd1_wbc_min',
 'd1_calcium_min',
 'd1_heartrate_min^2',
 'd1_sysbp_min^2',
 'd1_temp_min^2',
 'd1_temp_max^2',
 'd1_resprate_min^2',
 'd1_potassium_min^2',
 'd1_glucose_max^2',
 'd1_glucose_min^2',
 'd1_sodium_max^2',
 'd1_wbc_min^2',
 'd1_calcium_min^2',
 'h1_resprate_min',
 'd1_spo2_min',
 'd1_bun_min',
 'gcs_eyes_apache',
 'h1_resprate_max',
 'd1_sysbp_noninvasive_min',
 'gcs_verbal_apache',
 'd1_mbp_min',
 'age',
 'gcs_motor_apache',
 'resprate_apache',
 'd1_heartrate_max',
 'hospital_id',
 'icu_id',
 'ventilated_apache']

In [22]:
X_train.head()


,d1_heartrate_min,d1_sysbp_min,d1_temp_min,d1_temp_max,d1_resprate_min,d1_potassium_min,d1_glucose_max,d1_glucose_min,d1_sodium_max,d1_wbc_min,...,d1_sysbp_noninvasive_min,gcs_verbal_apache,d1_mbp_min,age,gcs_motor_apache,resprate_apache,d1_heartrate_max,hospital_id,icu_id,ventilated_apache
0,0.914919,0.390231,-0.092383,0.458270,-0.767431,0.656753,-0.208984,-0.039426,0.638455,0.239070,...,0.389046,0.653494,0.389635,-0.455229,0.417545,-1.123689,0.273893,161,413,1.0
1,-1.374299,-2.322143,-0.633739,-1.288626,-2.548470,1.926883,0.834819,1.680739,-0.019351,1.450874,...,-2.322253,0.653494,-1.078542,0.212808,0.417545,1.273162,-0.682486,109,429,0.0
2,1.091012,-0.675344,-1.581112,0.312695,0.419928,3.378460,1.665117,2.487066,0.419186,0.706885,...,-0.676107,0.007871,-1.525378,0.820113,-1.144717,0.274474,0.000642,21,513,1.0
3,0.327940,0.680843,-0.227722,0.458270,-0.371645,0.475306,-0.873223,-0.361957,-0.457889,0.854015,...,0.679542,0.653494,0.644970,-1.244726,0.417545,-1.123689,-0.363693,137,143,0.0
4,1.325804,1.891724,0.083557,-0.415178,0.024141,0.112412,-0.552965,-0.630732,0.419186,0.754539,...,1.889944,0.007871,0.964139,0.516461,0.417545,0.274474,1.412439,19,646,0.0


In [23]:
X_train.shape, X_val.shape, X_train_val.shape, X_test.shape


((107260, 37), (14674, 37), (134076, 37), (18343, 37))

In [24]:
dirname = "../data/preprocessed/12-01/"

for object, name in [
    (X_train, "X_train"),
    (X_train_2, "X_train_dt"),
    (y_train, "y_train"),
    (y_train_2, "y_train_dt"),
    (X_val, "X_val"),
    (X_val_2, "X_val_dt"),
    (y_val, "y_val"),
    (X_train_val, "X_train_val"),
    (X_train_val_2, "X_train_val_dt"),
    (y_train_val, "y_train_val"),
    (y_train_val_2, "y_train_val_dt"),
    (X_test, "X_test"),
    (X_test_2, "X_test_dt"),
    (y_test, "y_test"),
]:
    pathname = f"{dirname}{name}.csv"
    if not os.path.exists(dirname) or not os.path.exists(pathname):
        if not os.path.exists(dirname):
            os.makedirs(dirname)
        with open(pathname, "w") as f:
            object.to_csv(f, index=False)
